# 🏙️ Level 3: Multi-Feature Linear Regression

**[📖 Want a detailed explanation? Read the Manual (Streamlit App)](https://bookseal-seoul-apt-price-prediction.streamlit.app/Level_3_Multi_Features)**

In Level 2, we only used **Area**. But the **Location (District)** matters a lot!
Here, we learn how to add text data (categories) into our math formula using **One-Hot Encoding**.

### 1. Load Data
Load our standard dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error

url = "https://github.com/bookseal/seoul-apt-price-prediction/raw/main/data/sample.parquet"
df = pd.read_parquet(url)

print(f"Loaded {len(df):,} rows")
df.head()

### 2. Why One-Hot Encoding?
Computers only understand numbers. 
- **Bad Way**: Gangnam=1, Seocho=2, Nowon=3 (Implies order: 3 > 1?)
- **Good Way (One-Hot)**: Create independent Yes/No columns for each district.

Let's see how Scikit-Learn does this.

In [ ]:
# 1. Initialize Encoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# 2. Fit and Transform 'district' column
X_district = encoder.fit_transform(df[['district']])

# 3. See what happened
district_names = encoder.categories_[0]
print("District Columns:", district_names[:5], "...")
print("Shape of X_district:", X_district.shape)

# Show as DataFrame for clarity
pd.DataFrame(X_district, columns=district_names).head(5)

### 3. Prepare Training Data
We combine **Area** (Numerical) and **District** (One-Hot Encoded) into one big matrix `X`.

In [ ]:
X_area = df[['area_m2']].values

# Horizontal Stack (Paste side-by-side)
X = np.hstack([X_area, X_district])
y = df['price_10k_krw'].values

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Features Summary:")
print(f"1 Area Column + {X_district.shape[1]} District Columns = {X.shape[1]} Total Features")

### 4. Train Model
The code is exactly the same as Level 2! `model.fit(X, y)`.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Training Complete.")
print(f"Bias (Base Price): {model.intercept_:,.0f}")
print(f"Weight for Area: {model.coef_[0]:,.0f}")

### 5. Analyze District Weights
Which districts are expensive? Let's look at the learned weights.

In [ ]:
district_coefs = model.coef_[1:]  # Skip the first one (Area)

coef_df = pd.DataFrame({
    'District': district_names,
    'Effect': district_coefs
}).sort_values('Effect', ascending=False)

print("Top 5 Most Expensive Districts (Premium):")
display(coef_df.head(5))

print("\nTop 5 Cheapest Districts (Discount):")
display(coef_df.tail(5))

### 6. Evaluate Performance
Did adding 'District' help? Check RMSE.

In [ ]:
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Level 3 RMSE: {rmse:,.0f}")
print("Compare this to Level 2 (approx 42,000). The error dropped significantly!")

### 7. Interactive Prediction
Let's predict the price for a specific apartment.

In [ ]:
# Choose your inputs
my_area = 84
my_district = '강남구'

# Prepare input vector
d_vec = encoder.transform([[my_district]])
input_vec = np.hstack([[my_area], d_vec[0]]).reshape(1, -1)

pred = model.predict(input_vec)[0]

print(f"Predicted Price for {my_area}m² in {my_district}:")
print(f"{pred:,.0f} (10k KRW) = {pred/10000:.2f} 억원")